# 06 - 迁移学习**学习目标：**- 理解迁移学习的原理和优势- 掌握冻结层 vs 微调全部层的区别- 学会在自己的小数据集上微调预训练模型- 识别和应对过拟合---

## 1. 什么是迁移学习**核心思想**：用在大数据集上学到的知识，帮助在小数据集上的学习。```预训练模型 (COCO: 118k 图片, 80 类)    │    │  已学会：边缘、纹理、形状、物体部件...    ▼在你的数据集上微调 (几百张图片, 少数类别)    │    │  只需要学：你的数据集的特定特征    ▼高精度的定制模型```**为什么有效？**- 神经网络的底层特征（边缘、纹理）是通用的- 高层特征（物体部件、类别）才需要针对特定任务学习- 预训练 = 已经学会了通用特征

## 2. 迁移学习策略```策略 1: 只训练 Head（冻结 Backbone + Neck）┌──────────────────────────────────────┐│ Backbone │  Neck  │  Head  │         ││  ❄️冻结  │ ❄️冻结 │ 🔥训练 │  ← 最快 │└──────────────────────────────────────┘适用：数据很少（<100 张），任务与预训练相似策略 2: 全部微调（推荐）┌──────────────────────────────────────┐│ Backbone │  Neck  │  Head  │         ││ 🔥小学习率│🔥小lr │🔥大lr  │  ← 最佳 │└──────────────────────────────────────┘适用：数据中等（几百张），效果最好策略 3: 从零训练┌──────────────────────────────────────┐│ Backbone │  Neck  │  Head  │         ││  🔥训练  │ 🔥训练 │ 🔥训练 │  ← 最慢 │└──────────────────────────────────────┘适用：数据很大，或任务与预训练差异很大

## 3. 动手实验：微调预训练模型

In [ ]:
from ultralytics import YOLO# 加载预训练模型model = YOLO("yolo11n.pt")# 微调训练（ultralytics 默认就是迁移学习模式）results = model.train(    data="../configs/data/coco128.yaml",    epochs=20,    imgsz=640,    batch=16,    device="mps",    project="../outputs",    name="transfer_learning",        # 迁移学习关键参数    freeze=0,       # 0=全部训练（推荐），10=冻结前10层    lr0=0.01,       # 初始学习率    warmup_epochs=3, # warmup 轮数)

## 4. 冻结层实验对比不同冻结策略的效果：

In [ ]:
from ultralytics import YOLO

# 实验 1: 冻结 Backbone（前 10 层）
model1 = YOLO("yolo11n.pt")
r1 = model1.train(
    data="../configs/data/coco128.yaml",
    epochs=10,
    device="mps",
    project="../outputs",
    name="freeze_backbone",
    freeze=10,  # 冻结前 10 层
    verbose=False,
)
print(f"冻结 Backbone - mAP@0.5: {r1.results_dict.get('metrics/mAP50(B)', 0):.3f}")

In [ ]:
# 实验 2: 全部训练model2 = YOLO("yolo11n.pt")r2 = model2.train(    data="../configs/data/coco128.yaml",    epochs=10,    device="mps",    project="../outputs",    name="train_all",    freeze=0,   # 不冻结    verbose=False,)print(f"全部训练 - mAP@0.5: {r2.results_dict.get('metrics/mAP50(B)', 0):.3f}")

## 5. 过拟合识别与应对**过拟合的表现**：```Loss  │  │  train_loss ╲  │              ╲___________  ← train 持续下降  │                │  val_loss   ╲╱‾‾‾‾‾‾‾‾‾  ← val 先降后升  │              ↑  │           过拟合开始  └──────────────────────────→ epoch```**过拟合的应对方法**：| 方法 | 说明 ||------|------|| **更多数据** | 收集更多图片，或使用数据增强 || **早停 (Early Stopping)** | val_loss 不再下降时停止训练 || **减小模型** | 用更小的模型 (n → s → m) || **正则化** | 增加 weight_decay || **减少 epochs** | 不要训练太久 || **Dropout** | 随机关闭部分神经元 |

## 6. 练习### 练习 1: 对比不同冻结层数分别冻结 0、5、10、20 层，对比训练速度和最终 mAP。### 练习 2: 调整学习率冻结 Backbone 时，用更大学习率 (lr0=0.1) 训练 Head，观察效果。### 练习 3: 过拟合实验用极少数据（10 张图片）训练 100 轮，观察过拟合现象。

---**上一节**: [05 - 训练基础](05_training_basics.ipynb)**下一节**: [07 - 评估指标](07_evaluation_metrics.ipynb) - IoU、mAP、PR 曲线